In [ ]:
import kagglehub
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
from sklearn.impute import SimpleImputer

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
food_delivery_path = os.path.join(path, 'Q1_data.csv')
df_food_delivery = pd.read_csv(food_delivery_path)

In [ ]:
# Task 2: Write your code here:
df_food_delivery.head()

In [ ]:
# Task 3: Write your code here:
df_food_delivery.info()

In [ ]:
# Task 4: Write your code here:
df_food_delivery.describe()

In [ ]:
# Task 5: Write your code here:
# delivery_time distribution (target variable)
plt.figure(figsize=(10, 5))
plt.hist(df_food_delivery['Delivery_Time'].dropna(), bins=15, edgecolor='black')
plt.title('Delivery Time Distribution')
plt.xlabel('Delivery Time')
plt.ylabel('Frequency')
plt.show()

In [ ]:
# Task 1: Write your code here:
df_food_delivery.drop(columns=['Order_ID'])

In [ ]:
# Task 2: Write your code here:
# Missing values
print("Missing values:")
print(df_food_delivery.isnull().sum())


In [ ]:
num_cols = df_food_delivery.select_dtypes(include=['number']).columns
cat_cols = df_food_delivery.select_dtypes(include=['object', 'category']).columns

num_imputer = SimpleImputer(strategy='median')
cat_imputer = SimpleImputer(strategy='most_frequent')

# use .copy() to keep original df safe
df_clean = df_food_delivery.copy()

# Fill Missing Values
df_clean[num_cols] = num_imputer.fit_transform(df_food_delivery[num_cols])
df_clean[cat_cols] = cat_imputer.fit_transform(df_food_delivery[cat_cols])

print(f"Missing values after cleaning: {df_clean.isnull().sum().sum()}")
print(df_clean.isnull().sum())

In [ ]:
from numpy._core.defchararray import index
# Task 3: Write your code here:
# Duplicate Check

def check_duplicates(df):
  duplicates = df.duplicated().sum()
  print(f"Number of Duplicate Samples: {duplicates}")
  if duplicates > 0:
    print("Dropping Duplicates...")
    df.drop_duplicates(inplace=True)
    print("Duplicates Dropped.")
  else:
    print("No Duplicate Samples Found.")

check_duplicates(df_clean)

In [ ]:
# Task 4: Write your code here:
from sklearn.preprocessing import OneHotEncoder #import OneHotEncoder

print('data before encoding:\n', cat_cols) #show before encoding

onehot_encoder = OneHotEncoder(handle_unknown='ignore') # Instantiate OneHotEncoder
data_onehot_encoded = onehot_encoder.fit_transform() # Apply fit_transform to the copied

print('\nData after encoding:\n', data_onehot_encoded) #show after encoding

In [ ]:
# Task 5: Write your code here:
from sklearn.preprocessing import StandardScaler #import StandardScaler

print('data before scaling:\n', df_clean) #show before scaling
standard_scaler = StandardScaler() # Instantiate StandardScaler
data_standard_scaled = standard_scaler.fit_transform(df_clean) # Apply fit_transform

print('\nData after scaling:\n', data_standard_scaled) #show after scaling

In [ ]:
# Task 6: Write your code here:
import seaborn as sns

TARGET_COL = 'Delivery_Time'

plt.figure(figsize=(8, 5))
if df_clean[TARGET_COL].dtype == 'object' or df_clean[TARGET_COL].nunique() < 10:
    # Classification Problem
    sns.countplot(x=TARGET_COL, data=df_clean, palette='viridis')
    plt.title(f"Target Distribution: {TARGET_COL} (Classification)")
    plt.ylabel("Count")
else:
    # Regression Problem
    sns.histplot(df_clean[TARGET_COL], kde=True, color='blue')
    plt.title(f"Target Distribution: {TARGET_COL} (Regression)")
    plt.xlabel("Value")
plt.show()

# balanced

In [ ]:
# Task 1: Write your code here:
from sklearn.model_selection import train_test_split

# --- 1. Separate Features (X) and Target (y) ---
X = df_clean.drop(columns=[TARGET_COL])
y = df_clean[TARGET_COL]

# --- 2. Split Data ---
# Logic: If Classification -> Use Stratify. If Regression -> No Stratify.
is_classification = (y.dtype == 'object') or (y.nunique() < 20)

if is_classification:
    print("Detected Classification Task -> Using Stratified Split")
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )
else:
    print("Detected Regression Task -> Using Random Split")
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )

print(f"Train Shape: {X_train.shape}")
print(f"Test Shape:  {X_test.shape}")



In [ ]:
# Task 2,3,4,5: Write your code here:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split, StratifiedKFold, KFold
from sklearn.metrics import (mean_absolute_error, mean_squared_error, r2_score,
                             accuracy_score, precision_score, recall_score, f1_score,
                             classification_report, confusion_matrix)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

model = {}
model['Random Forest'] = RandomForestRegressor(n_estimators=100, random_state=42)

kfold = KFold(n_splits=5, shuffle=True, random_state=42)

mae_scores = []
rmse_scores = []

for train_idx, val_idx in kfold.split(X_train_scaled):
    X_fold_train, X_fold_val = X_train_scaled[train_idx], X_train_scaled[val_idx]
    y_fold_train, y_fold_val = y_train.iloc[train_idx], y_train.iloc[val_idx]

    # Train and predict
    model.fit(X_fold_train, y_fold_train)
    y_fold_pred = model.predict(X_fold_val)

    # Calculate metrics
    mae_scores.append(mean_absolute_error(y_fold_val, y_fold_pred))

mae_scores = np.array(mae_scores)

print(f"5-Fold CV Results:")
print(f"MAE:  ${mae_scores.mean():,.2f}")

In [ ]:
# Task 1: Write your code here:
importance = pd.DataFrame({
    'feature': num_cols + cat_cols,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 6))
plt.barh(importance['feature'], importance['importance'], color='purple')
plt.xlabel('Importance')
plt.title('Feature Importance')
plt.gca().invert_yaxis()
plt.show()

In [ ]:
# Task 2: Write your code here:

sns.histplot(df_clean['Delivery_Time'], kde=True, color='skyblue')



In [ ]:
# Task Bonus: Write your code here: